# 🌌 Colab Router — Qwen 3.8 27B on Free Colab T4 GPU
### 🕒 Dedicated Slot: **Multi-Shift Compatible** | 👤 Account: `Active Account`

Turn Google Colab (Tesla T4 GPU) into an **OpenAI-Compatible Streaming Inference Server** with 100% GPU VRAM offload.

### 📌 IMPORTANT — Multi-Account Setup Instructions:
1. **Check Logged In Account:** Ensure this tab is running under **`Active Account`** (check the top-right profile avatar).
2. **Save to Drive:** Click `File` ➔ `Save a copy in Drive` inside this account to keep your own independent notebook.
3. **Verify T4 GPU:** Click `Runtime` ➔ `Change runtime type` ➔ Ensure **T4 GPU** is selected (Python 3).
4. **Enter Tokens:** Add your `NGROK_AUTHTOKEN` in Cell 2 (or save in Colab 🔑 Secrets tab).
5. **Run All:** Press `Ctrl+F9` or click `Runtime` ➔ `Run all`.

---


In [ ]:
# ==============================================================================
# 🛡️ STEP 1: HARDWARE PRE-FLIGHT CHECK (Strict T4 GPU Verification)
# ==============================================================================
import sys, torch

print("🔍 Checking Hardware Accelerator...")
if not torch.cuda.is_available():
    error_banner = """
    ❌❌❌ FATAL: NO GPU DETECTED (Running on CPU) ❌❌❌
    Running 27B LLM on CPU will be 100x slower or crash with Out-Of-Memory.
    
    👉 HOW TO FIX RIGHT NOW:
       1. In top menu, click: Runtime ➔ Change runtime type
       2. Under Hardware accelerator, select 'T4 GPU'
       3. Click 'Save'
       4. Re-run notebook: Runtime ➔ Run all (Ctrl+F9)
    """
    print(error_banner, file=sys.stderr)
    raise RuntimeError("T4 GPU Required! Please enable GPU in Colab Runtime settings.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"🚀 ACTIVE ACCELERATOR: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
print("✅ GPU Gatekeeper Passed for Active Account! Ready for 100% VRAM offload.\n")


In [ ]:
# ==============================================================================
# ⚙️ STEP 2: CONFIGURATION & CREDENTIALS
# ==============================================================================
import os

# 🔑 Read from Colab Secrets tab (key icon on left) if configured, else use environment
try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    ngrok_domain = userdata.get('NGROK_DOMAIN')
    api_key = userdata.get('GDQ_API_KEY')
except Exception:
    ngrok_token = None
    ngrok_domain = None
    api_key = None

# 1. Ngrok Auth Token (paste your token here if not using Secrets)
os.environ["NGROK_AUTHTOKEN"] = ngrok_token or "YOUR_NGROK_AUTHTOKEN_HERE"

# 2. Ngrok Static Domain (optional)
os.environ["NGROK_DOMAIN"] = ngrok_domain or ""

# 3. API Key for gateway authorization
os.environ["GDQ_API_KEY"] = api_key or "CHANGE_ME"
if os.environ["GDQ_API_KEY"] == "CHANGE_ME":
    import secrets as _secrets
    os.environ["GDQ_API_KEY"] = "sk-" + _secrets.token_urlsafe(24)
    print("🔑 Generated Temporary API Key:", os.environ["GDQ_API_KEY"])

# 4. MODEL SELECTION (Default: Qwen 3.8 27B Uncensored - Maximum Capability)
os.environ["HF_MODEL_URL"] = "https://huggingface.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF/resolve/main/Qwen3.8-27B-Uncensored-IQ4_XS.gguf"
os.environ["MODEL_FILENAME"] = "Qwen3.8-27B-Uncensored-IQ4_XS.gguf"
os.environ["OLLAMA_MODEL_TAG"] = "qwen3.8-27b-uncensored"
os.environ["NUM_CTX"] = "4096"
os.environ["NUM_PREDICT"] = "4096"

print("✅ Configuration Loaded for Shift: Multi-Shift Compatible (Active Account)!")


In [ ]:
# ==============================================================================
# 🚀 STEP 3: LAUNCH ENGINE & HIGHWAY ROUTER
# ==============================================================================
import os, subprocess

GITHUB_REPO_URL = "https://github.com/agenticversestudio/colab-router.git"
repo_dir = "/content/colab-router"
scripts_ready = False

print(f"📦 Checking Colab Router engine repository...")
if os.path.exists(repo_dir):
    print("🔄 Pulling latest engine code...")
    subprocess.run(["git", "-C", repo_dir, "pull"], capture_output=True, text=True)
    os.system(f"cp {repo_dir}/colab/colab_payload.py /content/colab_payload.py")
    os.system(f"cp {repo_dir}/colab/hybrid_server.py /content/hybrid_server.py")
    scripts_ready = True
else:
    print(f"📦 Cloning from {GITHUB_REPO_URL}...")
    res = subprocess.run(["git", "clone", GITHUB_REPO_URL, repo_dir], capture_output=True, text=True)
    if res.returncode == 0:
        os.system(f"cp {repo_dir}/colab/colab_payload.py /content/colab_payload.py")
        os.system(f"cp {repo_dir}/colab/hybrid_server.py /content/hybrid_server.py")
        scripts_ready = True
    else:
        print(f"⚠️ Clone failed: {res.stderr.strip()}")

# Fallback to /content files if uploaded directly
if not scripts_ready and os.path.exists("/content/colab_payload.py") and os.path.exists("/content/hybrid_server.py"):
    scripts_ready = True

if not scripts_ready:
    raise FileNotFoundError("Unable to locate colab_payload.py. Please upload files to /content or check network access.")

print("\n🔥 Starting Sovereign Colab Router Autonomous Engine on Tesla T4...")
os.system("python3 /content/colab_payload.py")
